In [ ]:
# -*- coding: utf-8 -*-
from xtquant import xtdata
from xtquant.xttrader import XtQuantTrader, XtQuantTraderCallback
from xtquant.xttype import StockAccount
from xtquant import xtconstant
import random
import time
import math
import requests
class MyCallback(XtQuantTraderCallback):
    def on_disconnected(self):
        print("Disconnected")
    
    def on_stock_order(self, order):
        # 这里的回调可以在订单状态变化时收到通知
        print(f"订单状态更新: 订单ID {order.order_id}, 状态: {order.order_status}, 备注: {order.order_remark}")
# --- 微信发送函数 (纯文本，关键修改点1) ---
def send_wechat_text(key, content):
    """
    发送纯文本消息给企业微信机器人
    微信服务通知仅可靠支持纯文本格式。
    """
    url = f'https://qyapi.weixin.qq.com/cgi-bin/webhook/send?key={key}'
    headers = {'Content-Type': 'application/json'}
    data = {
        "msgtype": "text",  # 类型必须为"text"
        "text": {
            "content": content
        }
    }
    try:
        response = requests.post(url, headers=headers, json=data)
        res_json = response.json()
        if res_json['errcode'] == 0:
            print(">> 微信消息发送成功")
        else:
            print(f">> 发送失败: {res_json['errmsg']}")
    except Exception as e:
        print(f">> 发送异常: {e}")
def buy_stock_by_amount(trader, account, stock_code, target_amount):
    """
    根据金额下单函数
    :param trader: 交易对象
    :param account: 账户对象
    :param stock_code: 股票代码 (例如 '000001.SZ')
    :param target_amount: 计划买入金额 (例如 500)
    """
    
    # 1. 获取实时行情 (为了计算能买多少股，以及定什么价格)
    # 必须要先订阅或者下载一下实时行情，否则可能拿不到最新数据
    xtdata.subscribe_quote(stock_code, period='tick') 
    time.sleep(0.5) # 给一点点时间同步数据
    
    full_tick = xtdata.get_full_tick([stock_code])
    if not full_tick:
        print(f"错误: 无法获取 {stock_code} 的行情数据")
        return

    # 获取‘卖一价’ (askPrice) 或者是 '最新价' (lastPrice)
    # 使用卖一价下单成交概率更高
    tick_data = full_tick.get(stock_code)
    current_price = tick_data['lastPrice']
    
    # 简单的防错：如果价格为0或停牌
    if current_price <= 0:
        print(f"错误: {stock_code} 当前价格异常 ({current_price})，无法下单")
        return

    print(f"[{stock_code}] 当前价格: {current_price}")

    # 2. 计算股数
    # 逻辑：金额 / 价格，然后向下取整到100的倍数
    # 例如：500元 / 10元 = 50股 -> 不足100股 -> 结果为0，无法下单
    # 例如：5000元 / 10元 = 500股 -> 下单500股
    
    can_buy_shares = int(target_amount / current_price)
    # 向下取整到100的倍数 (A股最少买100股)
    buy_volume = (can_buy_shares // 100) * 100

    if buy_volume < 100:
        print(f"警告: 计划金额 {target_amount} 不足以来买入一手(100股) {stock_code}。")
        print(f"当前一手需要约 {current_price * 100} 元。")
        return

    # 3. 执行下单
    print(f"正在下单: 代码 {stock_code}, 价格 {current_price}, 数量 {buy_volume}, 预计金额 {current_price * buy_volume}")
    send_wechat_text(
        "238495d7-921a-4b33-8835-ce49cbe3835d", 
        f"正在下单: 代码 {stock_code}, 价格 {current_price}, 数量 {buy_volume}, 预计金额 {current_price * buy_volume}"
    )
    # order_stock 参数说明:
    # 账户, 股票代码, 交易类型(买入), 数量, 价格类型(限价), 价格, 策略名, 备注
    order_id = trader.order_stock(
        account,
        stock_code,
        xtconstant.STOCK_BUY, 
        buy_volume,
        xtconstant.FIX_PRICE,
        current_price,
        "strategy_buy_amount",
        "Python下单"
    )
    
    print(f"下单指令已发送，本地订单ID: {order_id}")

if __name__ == "__main__":
    # --- 1. 连接设置 (沿用你的配置) ---
    path = r"E:\Program_Files\中金财富QMT个人版交易端\userdata_mini"
    session_id = random.randint(10000, 99999)
    trader = XtQuantTrader(path, session_id)
    
    trader.register_callback(MyCallback())
    trader.start()
    
    ret = trader.connect()
    if ret == 0:
        print("QMT连接成功")
    else:
        print("QMT连接失败")
        exit()

    # --- 2. 账户设置 (你的账号) ---
    acc = StockAccount("8031048706", "STOCK")
    trader.subscribe(acc)

    # --- 3. 执行“按金额下单”逻辑 ---
    
    # 注意：A股代码通常需要后缀，深市 00.SZ，沪市 60.SH
    # 平安银行是 000001.SZ
    target_stock = "000001.SZ"
    
    # 设定金额：比如 500元
    # 注意：平安银行股价约 11元左右，100股需要1100元。
    # 500元是不够买1手的，代码里会打印警告。你可以尝试把金额改成 2000 测试。
    target_money = 2000 
    
    buy_stock_by_amount(trader, acc, target_stock, target_money)


In [7]:
# -*- coding: utf-8 -*-
from xtquant import xtdata
from xtquant.xttrader import XtQuantTrader, XtQuantTraderCallback
from xtquant.xttype import StockAccount
from xtquant import xtconstant
import random
import time
import math
import requests

class MyCallback(XtQuantTraderCallback):
    def on_disconnected(self):
        print("Disconnected")
    
    def on_stock_order(self, order):
        # 这里的回调可以在订单状态变化时收到通知
        print(f"订单状态更新: 订单ID {order.order_id}, 状态: {order.order_status}, 备注: {order.order_remark}")

# --- 微信发送函数 (纯文本，关键修改点1) ---
def send_wechat_text(key, content):
    """
    发送纯文本消息给企业微信机器人
    微信服务通知仅可靠支持纯文本格式。
    """
    url = f'https://qyapi.weixin.qq.com/cgi-bin/webhook/send?key={key}'
    headers = {'Content-Type': 'application/json'}
    data = {
        "msgtype": "text",  # 类型必须为"text"
        "text": {
            "content": content
        }
    }
    try:
        response = requests.post(url, headers=headers, json=data)
        res_json = response.json()
        if res_json['errcode'] == 0:
            print(">> 微信消息发送成功")
        else:
            print(f">> 发送失败: {res_json['errmsg']}")
    except Exception as e:
        print(f">> 发送异常: {e}")

def sell_stock_by_ratio(trader, account, stock_code, sell_ratio):
    """
    根据持股比例卖出函数
    :param trader: 交易对象
    :param account: 账户对象
    :param stock_code: 股票代码 (例如 '000001.SZ')
    :param sell_ratio: 卖出比例 (0-1之间的浮点数，例如0.5表示卖出50%)
    """
    
    # 参数检查
    if sell_ratio <= 0:
        print(f"错误: 卖出比例必须大于0，当前比例: {sell_ratio}")
        return
    
    if sell_ratio > 1:
        sell_ratio = 1  # 如果比例大于1，按100%处理
        print(f"警告: 卖出比例大于1，自动调整为1 (100%)")
    
    # 1. 获取账户持仓信息
    print(f"正在获取 {stock_code} 的持仓信息...")
    
    # 查询账户持仓
    positions = trader.query_stock_positions(account)
    
    if positions is None or len(positions) == 0:
        print(f"账户没有持仓信息")
        return
    
    # 查找目标股票的持仓
    target_position = None
    for pos in positions:
        if pos.stock_code == stock_code:
            target_position = pos
            break
    
    if target_position is None:
        print(f"未找到 {stock_code} 的持仓")
        return
    
    # 检查持仓数量
    if target_position.volume <= 0:
        print(f"{stock_code} 持仓数量为0，无法卖出")
        return
    
    print(f"股票 {stock_code} 当前持仓:")
    print(f"  持仓数量: {target_position.volume} 股")
    print(f"  可用数量: {target_position.can_use_volume} 股")
    print(f"  成本价: {target_position.open_price}")
    print(f"  当前价: {target_position.market_value / target_position.volume if target_position.volume > 0 else 0}")
    print(f"  总市值: {target_position.market_value}")
    
    # 2. 计算卖出数量
    # 使用可用数量进行计算
    available_shares = target_position.can_use_volume
    if available_shares <= 0:
        print(f"错误: {stock_code} 可用数量为0，无法卖出")
        return
    
    # 按比例计算卖出数量
    sell_volume = int(available_shares * sell_ratio)
    
    # 确保卖出数量是100的倍数（A股要求）
    sell_volume = (sell_volume // 100) * 100
    
    if sell_volume < 100:
        print(f"警告: 按比例 {sell_ratio*100:.1f}% 计算出的卖出数量不足100股")
        print(f"可用数量: {available_shares}股, 计算数量: {int(available_shares * sell_ratio)}股")
        
        # 如果可用数量本身就不足100股，考虑全仓卖出
        if available_shares < 100:
            print(f"可用数量 {available_shares} 股不足100股，将全部卖出")
            sell_volume = available_shares
        else:
            # 尝试最小卖出100股
            sell_volume = 100
            print(f"将按最小单位卖出: 100股")
    
    # 3. 获取实时行情确定卖出价格
    xtdata.subscribe_quote(stock_code, period='tick') 
    time.sleep(0.5) # 给一点点时间同步数据
    
    full_tick = xtdata.get_full_tick([stock_code])
    if not full_tick:
        print(f"错误: 无法获取 {stock_code} 的行情数据")
        return

    tick_data = full_tick.get(stock_code)
    if 'lastPrice' in tick_data and tick_data['lastPrice'] > 0:
        current_price = tick_data['lastPrice']
    elif 'bidPrice' in tick_data and len(tick_data['bidPrice']) > 0:
        # 使用买一价，更容易成交
        current_price = tick_data['bidPrice'][0]
    else:
        print(f"错误: {stock_code} 当前价格异常，无法获取有效价格")
        return
    
    # 4. 检查卖出数量是否超过可用数量
    if sell_volume > available_shares:
        print(f"警告: 计算卖出数量 {sell_volume} 超过可用数量 {available_shares}，调整为可用数量")
        sell_volume = available_shares
    
    print(f"[{stock_code}] 卖出信息:")
    print(f"  卖出比例: {sell_ratio*100:.1f}%")
    print(f"  当前价格: {current_price}")
    print(f"  卖出数量: {sell_volume} 股")
    print(f"  预计金额: {current_price * sell_volume:.2f} 元")
    
    # 5. 执行卖出订单
    send_wechat_text(
        "238495d7-921a-4b33-8835-ce49cbe3835d", 
        f"正在卖出: 代码 {stock_code}, 价格 {current_price}, 数量 {sell_volume}, 比例 {sell_ratio*100:.1f}%, 预计金额 {current_price * sell_volume:.2f}"
    )
    
    # 使用限价单卖出，使用买一价更容易成交
    order_id = trader.order_stock(
        account,
        stock_code,
        xtconstant.STOCK_SELL,  # 卖出操作
        sell_volume,
        xtconstant.FIX_PRICE,
        current_price,
        "strategy_sell_ratio",
        f"按比例卖出 {sell_ratio*100:.1f}%"
    )
    
    print(f"卖出指令已发送，本地订单ID: {order_id}")
    return order_id

# 原来的买入函数保持不变
def buy_stock_by_amount(trader, account, stock_code, target_amount):
    """
    根据金额下单函数
    :param trader: 交易对象
    :param account: 账户对象
    :param stock_code: 股票代码 (例如 '000001.SZ')
    :param target_amount: 计划买入金额 (例如 500)
    """
    
    # 1. 获取实时行情 (为了计算能买多少股，以及定什么价格)
    # 必须要先订阅或者下载一下实时行情，否则可能拿不到最新数据
    xtdata.subscribe_quote(stock_code, period='tick') 
    time.sleep(0.5) # 给一点点时间同步数据
    
    full_tick = xtdata.get_full_tick([stock_code])
    if not full_tick:
        print(f"错误: 无法获取 {stock_code} 的行情数据")
        return

    # 获取‘卖一价’ (askPrice) 或者是 '最新价' (lastPrice)
    # 使用卖一价下单成交概率更高
    tick_data = full_tick.get(stock_code)
    current_price = tick_data['lastPrice']
    
    # 简单的防错：如果价格为0或停牌
    if current_price <= 0:
        print(f"错误: {stock_code} 当前价格异常 ({current_price})，无法下单")
        return

    print(f"[{stock_code}] 当前价格: {current_price}")

    # 2. 计算股数
    # 逻辑：金额 / 价格，然后向下取整到100的倍数
    # 例如：500元 / 10元 = 50股 -> 不足100股 -> 结果为0，无法下单
    # 例如：5000元 / 10元 = 500股 -> 下单500股
    
    can_buy_shares = int(target_amount / current_price)
    # 向下取整到100的倍数 (A股最少买100股)
    buy_volume = (can_buy_shares // 100) * 100

    if buy_volume < 100:
        print(f"警告: 计划金额 {target_amount} 不足以来买入一手(100股) {stock_code}。")
        print(f"当前一手需要约 {current_price * 100} 元。")
        return

    # 3. 执行下单
    print(f"正在下单: 代码 {stock_code}, 价格 {current_price}, 数量 {buy_volume}, 预计金额 {current_price * buy_volume}")
    send_wechat_text(
        "238495d7-921a-4b33-8835-ce49cbe3835d", 
        f"正在下单: 代码 {stock_code}, 价格 {current_price}, 数量 {buy_volume}, 预计金额 {current_price * buy_volume}"
    )
    # order_stock 参数说明:
    # 账户, 股票代码, 交易类型(买入), 数量, 价格类型(限价), 价格, 策略名, 备注
    order_id = trader.order_stock(
        account,
        stock_code,
        xtconstant.STOCK_BUY, 
        buy_volume,
        xtconstant.FIX_PRICE,
        current_price,
        "strategy_buy_amount",
        "Python下单"
    )
    
    print(f"下单指令已发送，本地订单ID: {order_id}")

if __name__ == "__main__":
    # --- 1. 连接设置 (沿用你的配置) ---
    path = r"E:\Program_Files\中金财富QMT个人版交易端\userdata_mini"
    session_id = random.randint(10000, 99999)
    trader = XtQuantTrader(path, session_id)
    
    trader.register_callback(MyCallback())
    trader.start()
    
    ret = trader.connect()
    if ret == 0:
        print("QMT连接成功")
    else:
        print("QMT连接失败")
        exit()

    # --- 2. 账户设置 (你的账号) ---
    acc = StockAccount("8031048706", "STOCK")
    trader.subscribe(acc)

    # --- 3. 测试卖出函数 ---
    
    # 测试卖出：卖出平安银行50%的持仓
    target_stock = "002157.SZ"
    
    # 设置卖出比例 (0.5表示卖出50%)
    sell_ratio = 1  
    
    # 执行卖出
    sell_stock_by_ratio(trader, acc, target_stock, sell_ratio)
    
    # 如果想测试全仓卖出，可以使用 sell_ratio = 1
    # sell_stock_by_ratio(trader, acc, target_stock, 1)  # 卖出100%
    target_stock = "001318.SZ"
    
    # 设定金额：比如 500元
    # 注意：平安银行股价约 11元左右，100股需要1100元。
    # 500元是不够买1手的，代码里会打印警告。你可以尝试把金额改成 2000 测试。
    target_money = 1670 
    
    buy_stock_by_amount(trader, acc, target_stock, target_money)

QMT连接成功
正在获取 002157.SZ 的持仓信息...
002157.SZ 持仓数量为0，无法卖出
[001318.SZ] 当前价格: 16.7
正在下单: 代码 001318.SZ, 价格 16.7, 数量 100, 预计金额 1670.0
>> 微信消息发送成功
下单指令已发送，本地订单ID: 403701770
订单状态更新: 订单ID 403701770, 状态: 50, 备注: Python下单订单状态更新: 订单ID 403701770, 状态: 50, 备注: Python下单
订单状态更新: 订单ID 403701770, 状态: 50, 备注: Python下单
订单状态更新: 订单ID 403701770, 状态: 50, 备注: Python下单
订单状态更新: 订单ID 403701770, 状态: 50, 备注: Python下单
订单状态更新: 订单ID 403701770, 状态: 50, 备注: Python下单

订单状态更新: 订单ID 403701770, 状态: 50, 备注: Python下单


订单状态更新: 订单ID 403701770, 状态: 56, 备注: Python下单订单状态更新: 订单ID 403701770, 状态: 56, 备注: Python下单
订单状态更新: 订单ID 403701770, 状态: 56, 备注: Python下单
订单状态更新: 订单ID 403701770, 状态: 56, 备注: Python下单
订单状态更新: 订单ID 403701770, 状态: 56, 备注: Python下单

订单状态更新: 订单ID 403701770, 状态: 56, 备注: Python下单
订单状态更新: 订单ID 403701770, 状态: 56, 备注: Python下单


In [6]:
# -*- coding: utf-8 -*-
from xtquant import xtdata
from xtquant.xttrader import XtQuantTrader, XtQuantTraderCallback
from xtquant.xttype import StockAccount
from xtquant import xtconstant
import random
import time
import math
import requests

class MyCallback(XtQuantTraderCallback):
    def on_disconnected(self):
        print("Disconnected")
    
    def on_stock_order(self, order):
        # 这里的回调可以在订单状态变化时收到通知
        print(f"订单状态更新: 订单ID {order.order_id}, 状态: {order.order_status}, 备注: {order.order_remark}")

# --- 微信发送函数 (纯文本，关键修改点1) ---
def send_wechat_text(key, content):
    """
    发送纯文本消息给企业微信机器人
    微信服务通知仅可靠支持纯文本格式。
    """
    url = f'https://qyapi.weixin.qq.com/cgi-bin/webhook/send?key={key}'
    headers = {'Content-Type': 'application/json'}
    data = {
        "msgtype": "text",  # 类型必须为"text"
        "text": {
            "content": content
        }
    }
    try:
        response = requests.post(url, headers=headers, json=data)
        res_json = response.json()
        if res_json['errcode'] == 0:
            print(">> 微信消息发送成功")
        else:
            print(f">> 发送失败: {res_json['errmsg']}")
    except Exception as e:
        print(f">> 发送异常: {e}")

def buy_stock_by_price_and_lots(trader, account, stock_code, price, lots):
    """
    按指定价格和手数买入函数
    :param trader: 交易对象
    :param account: 账户对象
    :param stock_code: 股票代码 (例如 '000001.SZ')
    :param price: 指定买入价格 (元)
    :param lots: 买入手数 (1手=100股)
    """
    
    # 参数检查
    if price <= 0:
        print(f"错误: 买入价格必须大于0，当前价格: {price}")
        return
    
    if lots <= 0:
        print(f"错误: 买入手数必须大于0，当前手数: {lots}")
        return
    
    # 计算买入数量 (1手=100股)
    buy_volume = lots * 100
    
    # 1. 获取实时行情验证价格合理性（可选）
    xtdata.subscribe_quote(stock_code, period='tick') 
    time.sleep(0.5)
    
    full_tick = xtdata.get_full_tick([stock_code])
    if full_tick:
        tick_data = full_tick.get(stock_code)
        if 'lastPrice' in tick_data and tick_data['lastPrice'] > 0:
            current_price = tick_data['lastPrice']
            print(f"[{stock_code}] 当前市价: {current_price}, 指定买入价: {price}")
            
            # 价格合理性检查（可选的警告）
            if price > current_price * 1.1:  # 如果指定价格比市价高10%
                print(f"警告: 指定买入价 {price} 比当前市价 {current_price} 高出 {((price/current_price)-1)*100:.2f}%")
                print("这可能导致无法成交或高价买入")
        else:
            print(f"无法获取 {stock_code} 的实时价格")
    else:
        print(f"无法获取 {stock_code} 的行情数据")
    
    # 2. 计算预计金额
    estimated_amount = price * buy_volume
    
    print(f"买入订单详情:")
    print(f"  股票代码: {stock_code}")
    print(f"  买入价格: {price} 元")
    print(f"  买入手数: {lots} 手")
    print(f"  买入数量: {buy_volume} 股")
    print(f"  预计金额: {estimated_amount:.2f} 元")
    
    # 3. 执行买入订单
    send_wechat_text(
        "238495d7-921a-4b33-8835-ce49cbe3835d", 
        f"按指定价格买入: 代码 {stock_code}, 价格 {price}, 手数 {lots}, 数量 {buy_volume}, 预计金额 {estimated_amount:.2f}"
    )
    
    order_id = trader.order_stock(
        account,
        stock_code,
        xtconstant.STOCK_BUY,
        buy_volume,
        xtconstant.FIX_PRICE,  # 限价单
        price,
        "strategy_buy_fixed",
        f"指定价格 {price} 买入 {lots} 手"
    )
    
    print(f"买入指令已发送，本地订单ID: {order_id}")
    return order_id

def sell_stock_by_price_and_lots(trader, account, stock_code, price, lots):
    """
    按指定价格和手数卖出函数
    :param trader: 交易对象
    :param account: 账户对象
    :param stock_code: 股票代码 (例如 '000001.SZ')
    :param price: 指定卖出价格 (元)
    :param lots: 卖出手数 (1手=100股)
    """
    
    # 参数检查
    if price <= 0:
        print(f"错误: 卖出价格必须大于0，当前价格: {price}")
        return
    
    if lots <= 0:
        print(f"错误: 卖出手数必须大于0，当前手数: {lots}")
        return
    
    # 计算卖出数量 (1手=100股)
    sell_volume = lots * 100
    
    # 1. 检查持仓是否足够
    positions = trader.query_stock_positions(account)
    if positions is None or len(positions) == 0:
        print(f"账户没有持仓信息")
        return
    
    # 查找目标股票的持仓
    target_position = None
    for pos in positions:
        if pos.stock_code == stock_code:
            target_position = pos
            break
    
    if target_position is None:
        print(f"未找到 {stock_code} 的持仓")
        return
    
    # 检查可用数量
    available_shares = target_position.can_use_volume
    if available_shares <= 0:
        print(f"错误: {stock_code} 可用数量为0，无法卖出")
        return
    
    if sell_volume > available_shares:
        print(f"错误: 计划卖出 {sell_volume} 股，但可用数量只有 {available_shares} 股")
        
        # 计算最大可卖手数
        max_lots = available_shares // 100
        if max_lots > 0:
            print(f"最大可卖 {max_lots} 手 ({max_lots*100} 股)")
            choice = input(f"是否调整为最大可卖 {max_lots} 手？(y/n): ")
            if choice.lower() == 'y':
                sell_volume = max_lots * 100
                lots = max_lots
            else:
                return
        else:
            print(f"可用数量不足100股，无法卖出")
            return
    
    # 2. 获取实时行情验证价格合理性（可选）
    xtdata.subscribe_quote(stock_code, period='tick') 
    time.sleep(0.5)
    
    full_tick = xtdata.get_full_tick([stock_code])
    if full_tick:
        tick_data = full_tick.get(stock_code)
        if 'lastPrice' in tick_data and tick_data['lastPrice'] > 0:
            current_price = tick_data['lastPrice']
            print(f"[{stock_code}] 当前市价: {current_price}, 指定卖出价: {price}")
            
            # 价格合理性检查
            if price < current_price * 0.9:  # 如果指定价格比市价低10%
                print(f"警告: 指定卖出价 {price} 比当前市价 {current_price} 低 {((1-price/current_price))*100:.2f}%")
                print("这可能导致低价卖出")
        else:
            print(f"无法获取 {stock_code} 的实时价格")
    else:
        print(f"无法获取 {stock_code} 的行情数据")
    
    # 3. 计算预计金额
    estimated_amount = price * sell_volume
    
    print(f"卖出订单详情:")
    print(f"  股票代码: {stock_code}")
    print(f"  当前持仓: {target_position.volume} 股")
    print(f"  可用数量: {available_shares} 股")
    print(f"  卖出价格: {price} 元")
    print(f"  卖出手数: {lots} 手")
    print(f"  卖出数量: {sell_volume} 股")
    print(f"  预计金额: {estimated_amount:.2f} 元")
    print(f"  剩余数量: {available_shares - sell_volume} 股")
    
    # 4. 执行卖出订单
    send_wechat_text(
        "238495d7-921a-4b33-8835-ce49cbe3835d", 
        f"按指定价格卖出: 代码 {stock_code}, 价格 {price}, 手数 {lots}, 数量 {sell_volume}, 预计金额 {estimated_amount:.2f}"
    )
    
    order_id = trader.order_stock(
        account,
        stock_code,
        xtconstant.STOCK_SELL,
        sell_volume,
        xtconstant.FIX_PRICE,  # 限价单
        price,
        "strategy_sell_fixed",
        f"指定价格 {price} 卖出 {lots} 手"
    )
    
    print(f"卖出指令已发送，本地订单ID: {order_id}")
    return order_id

def buy_stock_by_amount(trader, account, stock_code, target_amount):
    """
    根据金额下单函数
    :param trader: 交易对象
    :param account: 账户对象
    :param stock_code: 股票代码 (例如 '000001.SZ')
    :param target_amount: 计划买入金额 (例如 500)
    """
    
    # 1. 获取实时行情 (为了计算能买多少股，以及定什么价格)
    # 必须要先订阅或者下载一下实时行情，否则可能拿不到最新数据
    xtdata.subscribe_quote(stock_code, period='tick') 
    time.sleep(0.5) # 给一点点时间同步数据
    
    full_tick = xtdata.get_full_tick([stock_code])
    if not full_tick:
        print(f"错误: 无法获取 {stock_code} 的行情数据")
        return

    # 获取‘卖一价’ (askPrice) 或者是 '最新价' (lastPrice)
    # 使用卖一价下单成交概率更高
    tick_data = full_tick.get(stock_code)
    current_price = tick_data['lastPrice']
    
    # 简单的防错：如果价格为0或停牌
    if current_price <= 0:
        print(f"错误: {stock_code} 当前价格异常 ({current_price})，无法下单")
        return

    print(f"[{stock_code}] 当前价格: {current_price}")

    # 2. 计算股数
    # 逻辑：金额 / 价格，然后向下取整到100的倍数
    # 例如：500元 / 10元 = 50股 -> 不足100股 -> 结果为0，无法下单
    # 例如：5000元 / 10元 = 500股 -> 下单500股
    
    can_buy_shares = int(target_amount / current_price)
    # 向下取整到100的倍数 (A股最少买100股)
    buy_volume = (can_buy_shares // 100) * 100

    if buy_volume < 100:
        print(f"警告: 计划金额 {target_amount} 不足以来买入一手(100股) {stock_code}。")
        print(f"当前一手需要约 {current_price * 100} 元。")
        return

    # 3. 执行下单
    print(f"正在下单: 代码 {stock_code}, 价格 {current_price}, 数量 {buy_volume}, 预计金额 {current_price * buy_volume}")
    send_wechat_text(
        "238495d7-921a-4b33-8835-ce49cbe3835d", 
        f"正在下单: 代码 {stock_code}, 价格 {current_price}, 数量 {buy_volume}, 预计金额 {current_price * buy_volume}"
    )
    # order_stock 参数说明:
    # 账户, 股票代码, 交易类型(买入), 数量, 价格类型(限价), 价格, 策略名, 备注
    order_id = trader.order_stock(
        account,
        stock_code,
        xtconstant.STOCK_BUY, 
        buy_volume,
        xtconstant.FIX_PRICE,
        current_price,
        "strategy_buy_amount",
        "Python下单"
    )
    
    print(f"下单指令已发送，本地订单ID: {order_id}")

def sell_stock_by_ratio(trader, account, stock_code, sell_ratio):
    """
    根据持股比例卖出函数
    :param trader: 交易对象
    :param account: 账户对象
    :param stock_code: 股票代码 (例如 '000001.SZ')
    :param sell_ratio: 卖出比例 (0-1之间的浮点数，例如0.5表示卖出50%)
    """
    
    # 参数检查
    if sell_ratio <= 0:
        print(f"错误: 卖出比例必须大于0，当前比例: {sell_ratio}")
        return
    
    if sell_ratio > 1:
        sell_ratio = 1  # 如果比例大于1，按100%处理
        print(f"警告: 卖出比例大于1，自动调整为1 (100%)")
    
    # 1. 获取账户持仓信息
    print(f"正在获取 {stock_code} 的持仓信息...")
    
    # 查询账户持仓
    positions = trader.query_stock_positions(account)
    
    if positions is None or len(positions) == 0:
        print(f"账户没有持仓信息")
        return
    
    # 查找目标股票的持仓
    target_position = None
    for pos in positions:
        if pos.stock_code == stock_code:
            target_position = pos
            break
    
    if target_position is None:
        print(f"未找到 {stock_code} 的持仓")
        return
    
    # 检查持仓数量
    if target_position.volume <= 0:
        print(f"{stock_code} 持仓数量为0，无法卖出")
        return
    
    print(f"股票 {stock_code} 当前持仓:")
    print(f"  持仓数量: {target_position.volume} 股")
    print(f"  可用数量: {target_position.can_use_volume} 股")
    print(f"  成本价: {target_position.open_price}")
    print(f"  当前价: {target_position.market_value / target_position.volume if target_position.volume > 0 else 0}")
    print(f"  总市值: {target_position.market_value}")
    
    # 2. 计算卖出数量
    # 使用可用数量进行计算
    available_shares = target_position.can_use_volume
    if available_shares <= 0:
        print(f"错误: {stock_code} 可用数量为0，无法卖出")
        return
    
    # 按比例计算卖出数量
    sell_volume = int(available_shares * sell_ratio)
    
    # 确保卖出数量是100的倍数（A股要求）
    sell_volume = (sell_volume // 100) * 100
    
    if sell_volume < 100:
        print(f"警告: 按比例 {sell_ratio*100:.1f}% 计算出的卖出数量不足100股")
        print(f"可用数量: {available_shares}股, 计算数量: {int(available_shares * sell_ratio)}股")
        
        # 如果可用数量本身就不足100股，考虑全仓卖出
        if available_shares < 100:
            print(f"可用数量 {available_shares} 股不足100股，将全部卖出")
            sell_volume = available_shares
        else:
            # 尝试最小卖出100股
            sell_volume = 100
            print(f"将按最小单位卖出: 100股")
    
    # 3. 获取实时行情确定卖出价格
    xtdata.subscribe_quote(stock_code, period='tick') 
    time.sleep(0.5) # 给一点点时间同步数据
    
    full_tick = xtdata.get_full_tick([stock_code])
    if not full_tick:
        print(f"错误: 无法获取 {stock_code} 的行情数据")
        return

    tick_data = full_tick.get(stock_code)
    if 'lastPrice' in tick_data and tick_data['lastPrice'] > 0:
        current_price = tick_data['lastPrice']
    elif 'bidPrice' in tick_data and len(tick_data['bidPrice']) > 0:
        # 使用买一价，更容易成交
        current_price = tick_data['bidPrice'][0]
    else:
        print(f"错误: {stock_code} 当前价格异常，无法获取有效价格")
        return
    
    # 4. 检查卖出数量是否超过可用数量
    if sell_volume > available_shares:
        print(f"警告: 计算卖出数量 {sell_volume} 超过可用数量 {available_shares}，调整为可用数量")
        sell_volume = available_shares
    
    print(f"[{stock_code}] 卖出信息:")
    print(f"  卖出比例: {sell_ratio*100:.1f}%")
    print(f"  当前价格: {current_price}")
    print(f"  卖出数量: {sell_volume} 股")
    print(f"  预计金额: {current_price * sell_volume:.2f} 元")
    
    # 5. 执行卖出订单
    send_wechat_text(
        "238495d7-921a-4b33-8835-ce49cbe3835d", 
        f"正在卖出: 代码 {stock_code}, 价格 {current_price}, 数量 {sell_volume}, 比例 {sell_ratio*100:.1f}%, 预计金额 {current_price * sell_volume:.2f}"
    )
    
    # 使用限价单卖出，使用买一价更容易成交
    order_id = trader.order_stock(
        account,
        stock_code,
        xtconstant.STOCK_SELL,  # 卖出操作
        sell_volume,
        xtconstant.FIX_PRICE,
        current_price,
        "strategy_sell_ratio",
        f"按比例卖出 {sell_ratio*100:.1f}%"
    )
    
    print(f"卖出指令已发送，本地订单ID: {order_id}")
    return order_id

if __name__ == "__main__":
    # --- 1. 连接设置 (沿用你的配置) ---
    path = r"E:\Program_Files\中金财富QMT个人版交易端\userdata_mini"
    session_id = random.randint(10000, 99999)
    trader = XtQuantTrader(path, session_id)
    
    trader.register_callback(MyCallback())
    trader.start()
    
    ret = trader.connect()
    if ret == 0:
        print("QMT连接成功")
    else:
        print("QMT连接失败")
        exit()

    # --- 2. 账户设置 (你的账号) ---
    acc = StockAccount("8031048706", "STOCK")
    trader.subscribe(acc)

    # --- 3. 测试新增的指定价格和手数下单方法 ---
    
    target_stock = "002198.SZ"  # 东方通信
    
    # 测试1: 按指定价格和手数买入
    print("\n=== 测试1: 按指定价格和手数买入 ===")
    buy_price = 7.82  # 指定买入价格
    buy_lots = 5       # 买入5手（500股）
    # buy_stock_by_price_and_lots(trader, acc, target_stock, buy_price, buy_lots)
    
    # 测试2: 按指定价格和手数卖出
    print("\n=== 测试2: 按指定价格和手数卖出 ===")
    sell_price = 13.17 # 指定卖出价格
    sell_lots = 3       # 卖出2手（200股）
    # sell_stock_by_price_and_lots(trader, acc, target_stock, sell_price, sell_lots)
    
    # 测试3: 原按金额买入方法
    print("\n=== 测试3: 按金额买入 ===")
    target_amount = 4000  # 买入2000元
    buy_stock_by_amount(trader, acc, target_stock, target_amount)
    
    # # 测试4: 原按比例卖出方法
    # print("\n=== 测试4: 按比例卖出 ===")
    # sell_ratio = 0.5  # 卖出50%
    # # sell_stock_by_ratio(trader, acc, target_stock, sell_ratio)
    
    print("\n所有功能函数已定义完成，请根据需要取消注释进行测试")

QMT连接成功

=== 测试1: 按指定价格和手数买入 ===

=== 测试2: 按指定价格和手数卖出 ===

=== 测试3: 按金额买入 ===
[002198.SZ] 当前价格: 7.82
正在下单: 代码 002198.SZ, 价格 7.82, 数量 500, 预计金额 3910.0
>> 微信消息发送成功
下单指令已发送，本地订单ID: 940572678

所有功能函数已定义完成，请根据需要取消注释进行测试
订单状态更新: 订单ID 940572678, 状态: 50, 备注: Python下单订单状态更新: 订单ID 940572678, 状态: 50, 备注: Python下单
订单状态更新: 订单ID 940572678, 状态: 50, 备注: Python下单
订单状态更新: 订单ID 940572678, 状态: 50, 备注: Python下单
订单状态更新: 订单ID 940572678, 状态: 50, 备注: Python下单

订单状态更新: 订单ID 940572678, 状态: 50, 备注: Python下单


订单状态更新: 订单ID 940572678, 状态: 56, 备注: Python下单订单状态更新: 订单ID 940572678, 状态: 56, 备注: Python下单
订单状态更新: 订单ID 940572678, 状态: 56, 备注: Python下单
订单状态更新: 订单ID 940572678, 状态: 56, 备注: Python下单

订单状态更新: 订单ID 940572678, 状态: 56, 备注: Python下单
订单状态更新: 订单ID 940572678, 状态: 56, 备注: Python下单
